# USP at AmericasNLP 2026 — Stage 1 (Captioning) + Stage 2 (Translation)
**Platform:** Kaggle, NVIDIA T4

## Pipeline
1. **Stage 1** — Qwen3-VL-8B-Instruct generates a Spanish caption per image using a language-specific cultural prompt (zero-shot, no VLM fine-tuning)
2. **Stage 2** — Fine-tuned NLLB-200-distilled-600M translates Spanish → target Indigenous language

The notebook covers:
- Guaraní (full dev + test, including iterative prompt refinement)
- Maya, Wixárika, Nahuatl, Bribri (test set, via `run_language()`)

> **Note on vocabulary gap:** `bzd_Latn` (Bribri) and `yua_Latn` (Maya) are absent from the base NLLB-200 vocabulary — `tokenizer.convert_tokens_to_ids("bzd_Latn")` returns id 3 (`<s>`), causing the base model to silently output English. Fine-tuned models learn these tokens from training data.

## 1. Setup

In [ ]:
!pip install -q gdown transformers sentencepiece sacrebleu pillow

import os, torch, json, gc
import pandas as pd
from tqdm import tqdm
from PIL import Image as PILImage
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import CHRF

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

OUTPUT_DIR   = "/kaggle/working"
SPANISH_OUT  = f"{OUTPUT_DIR}/guarani_dev_spanish_captions.txt"
TRANSLATED_OUT = f"{OUTPUT_DIR}/guarani_dev_translated.txt

## 2. Clone task repository

In [ ]:
if not os.path.exists("/kaggle/working/americasnlp2026"):
    os.system("git clone https://github.com/AmericasNLP/americasnlp2026.git /kaggle/working/americasnlp2026")

# Load Guaraní dev set
df = pd.read_json("/kaggle/working/americasnlp2026/data/dev/guarani/guarani.jsonl", lines=True)
df["filename"] = df["filename"].str.split("data/guarani/").str[-1]
df["filepath"] = df["filename"].apply(
    lambda x: f"/kaggle/working/americasnlp2026/data/dev/guarani/{x}"
)
print(f"Loaded {len(df)} dev images")

# Quick sanity check on the JSONL format
with open("/kaggle/working/americasnlp2026/data/test/guarani/guarani.jsonl") as f:
    print(json.dumps(json.loads(f.readline()), indent=2, ensure_ascii=False))

## 3. Guaraní cultural prompts

Two prompt versions were tested on the dev set.
**V1** is the initial prompt; **V2** adds few-shot examples and an expanded cultural vocabulary list.
V2 was used for the final test set submission.

In [ ]:
# ── Prompt V1 (initial) ──────────────────────────────────────────────────────
GUARANI_TEMPLATE = """
Eres un sistema de subtitulado de imágenes diseñado para describir imágenes con relevancia cultural para el pueblo Guaraní de Paraguay.

Tu tarea: Generar subtítulos concisos, respetuosos y culturalmente precisos (2-4 oraciones máximo).

CONTEXTO CULTURAL A RECONOCER:

Gastronomía y Bebidas:
- Tereré: bebida fría de yerba mate con agua fría, bebida nacional de Paraguay
- Chipa: pan de almidón de mandioca y queso, alimento tradicional
- Sopa paraguaya: pastel de harina de maíz con queso y cebolla
- Mbejú: torta de almidón de mandioca
- Vori vori: sopa de bolitas de harina de maíz con queso
- Mandioca (mandi'o): tubérculo base de la alimentación

Artesanía y Arte:
- Ñandutí: encaje de araña, artesanía típica de Itauguá
- Ao po'i: tela bordada fina, tejido tradicional

Naturaleza y Flora:
- Jacarandá (tajy): árbol nacional de Paraguay, flores violetas
- Mburucuyá: flor de la pasión, flor nacional

Cultura y Sociedad:
- Opy: casa ceremonial Mbya Guaraní
- Danzas tradicionales: polka paraguaya, galopa
- Fiestas: San Juan (tatapyi, pelota tata)
- Caacupé: devoción a la Virgen

Mitología Guaraní:
- Jasy Jatere, Pombero, Kurupi, Luison

Arquitectura y Objetos:
- Tatakua: horno de barro tradicional
- Kambuchi: vasija de barro
- Guampa: recipiente para tereré
- Bombilla: filtro metálico para tereré
- Misiones jesuíticas: ruinas históricas

DIRECTRICES:
1. Máximo 2-4 oraciones
2. Identifica primero lo visible
3. Añade contexto cultural guaraní cuando sea relevante
4. NO generes emojis ni caracteres especiales
5. Escribe SOLO en español
"""

# ── Prompt V2 (final submission) — adds few-shot examples ─────────────────────
GUARANI_TEMPLATE_V2 = """
Eres un sistema de subtitulado de imágenes para el pueblo Guaraní de Paraguay.
Genera un subtítulo en ESPAÑOL, conciso y culturalmente preciso (2-4 oraciones).

EJEMPLOS DE SUBTÍTULOS CORRECTOS:

Imagen de encaje artesanal colorido:
"Ñandutí, encaje artesanal tradicional de Itauguá, Paraguay. Se teje a mano con hilos de colores formando patrones circulares como telas de araña. Es patrimonio cultural del Paraguay."

Imagen de sopa con bolitas amarillas:
"Vori vori, sopa tradicional paraguaya elaborada con bolitas de harina de maíz y queso. Es un plato típico de la gastronomía guaraní, especialmente consumido en invierno."

Imagen de vasija de barro con tela:
"Kambuchi, vasija de barro tradicional guaraní, utilizada para transportar y conservar agua fresca. Las mujeres la llevan sobre la cabeza usando un anillo de tela como apoyo."

CONTEXTO CULTURAL ESPECÍFICO DE PARAGUAY:
- Tereré: bebida de yerba mate fría — se toma con guampa y bombilla
- Chipa: pan de almidón de mandioca y queso
- Sopa paraguaya: pastel de maíz con queso y cebolla
- Ñandutí: encaje de araña de Itauguá — tejido circular con hilos de colores
- Tatakua: horno de barro para hacer chipa y pan
- Kambuchi: vasija de barro para agua
- Jopara: guiso de locro, poroto y carne — se come el 1 de octubre (Karai Octubre)
- Pohã ñana: hierbas medicinales guaraníes — Mercado 4, Asunción
- San Juan: fiesta del 24 de junio — pelota tata (bola de fuego), tatapyi (fogata)
- Caacupé: devoción a la Virgen — peregrinación el 8 de diciembre
- Misiones jesuíticas: ruinas de Jesús, Trinidad, San Ignacio
- Manduvi: maní/cacahuate paraguayo
- Pombero, Jasy Jatere, Kurupi: seres mitológicos guaraníes
- Ao po'i: tela bordada fina tradicional

DIRECTRICES:
1. Máximo 2-4 oraciones en español
2. Identifica el elemento cultural específico si lo reconoces
3. Si no reconoces el elemento, describe lo que ves claramente
4. NO uses emojis ni caracteres especiales
5. NO mezcles español con guaraní en la respuesta
"""
print("Prompts ready")

## 4. Guaraní dev set — Stage 1: VLM captioning

Generates Spanish captions for the 50 Guaraní dev images.
Resumes from the last saved line if interrupted.

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

model_name = "Qwen/Qwen3-VL-8B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
vlm = AutoModelForImageTextToText.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
print(f"VLM loaded | GPU used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Resume from existing progress
if os.path.exists(SPANISH_OUT):
    with open(SPANISH_OUT) as f:
        generated_captions = [l.strip() for l in f.readlines()]
    print(f"Resuming from {len(generated_captions)}/{len(df)}")
else:
    generated_captions = []
    print("Starting fresh")

start_idx = len(generated_captions)
torch.cuda.empty_cache()

for idx, row in tqdm(df.iloc[start_idx:].iterrows(),
                     total=len(df)-start_idx,
                     desc="Captioning dev"):
    try:
        img = PILImage.open(row["filepath"]).convert("RGB").resize((384, 384))
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": GUARANI_TEMPLATE_V2}
        ]}]
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = vlm.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                repetition_penalty=1.3
            )
        response_text = processor.decode(
            outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
        )
    except Exception as e:
        print(f"Error on {row['id']}: {e}")
        response_text = "Imagen cultural guaraní."

    generated_captions.append(response_text)
    del inputs, outputs
    torch.cuda.empty_cache()

    with open(SPANISH_OUT, "w", encoding="utf-8") as f:
        for caption in generated_captions:
            f.write(caption.replace("\n", " ") + "\n")

print(f"Done: {len(generated_captions)} captions saved to {SPANISH_OUT}")

## 5. Guaraní dev set — Stage 2: Translation and evaluation

Translates the Spanish captions to Guaraní using NLLB-200-distilled-600M.
Evaluates using ChrF++ (word_order=2) against the reference captions.

In [ ]:
# Free VLM before loading NMT
del vlm, processor
torch.cuda.empty_cache(); gc.collect()
print(f"GPU free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")

nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
).to("cuda")

with open(SPANISH_OUT) as f:
    spanish_captions = [l.strip() for l in f.readlines()]

translated_captions = []
for caption in tqdm(spanish_captions, desc="Translating"):
    inputs = nllb_tokenizer(
        caption, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")
    outputs = nllb_model.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("grn_Latn"),
        max_length=256
    )
    translated = nllb_tokenizer.decode(outputs[0], skip_special_tokens=True)
    translated_captions.append(translated)

    with open(TRANSLATED_OUT, "w", encoding="utf-8") as f:
        for t in translated_captions:
            f.write(t + "\n")

print(f"Done: {len(translated_captions)} translations saved")

In [ ]:
# Evaluate against dev set references
with open(TRANSLATED_OUT) as f:
    translated = [l.strip() for l in f.readlines()]

df["generated_caption"] = translated
df["chrf_score"] = df.apply(
    lambda r: CHRF(word_order=2).sentence_score(
        r["generated_caption"], [r["target_caption"]]
    ).score, axis=1
)

print(f"Mean ChrF++ (base NLLB): {df['chrf_score'].mean():.2f}")
print(f"Reference baseline (Sheffield): 20.82")

# Save full dev results
df.to_json(
    f"{OUTPUT_DIR}/guarani_dev_results.jsonl",
    orient="records", lines=True, force_ascii=False
)

# Show a few examples
for _, row in df.head(3).iterrows():
    print(f"\nID: {row['id']} | ChrF++: {row['chrf_score']:.2f}")
    print(f"  REF: {row['target_caption'][:100]}")
    print(f"  GEN: {row['generated_caption'][:100]}")

## 6. Guaraní dev set — Compare base vs fine-tuned NLLB

Evaluates the fine-tuned Guaraní model on the dev set and compares against
the base NLLB-200. The base model scored 19.49 ChrF++; fine-tuned scored 17.57.
The fine-tuned version was submitted for the test set (see Section 7).

In [ ]:
# Download the fine-tuned Guaraní model from Drive
import subprocess
FOLDER_ID = "1YtRnDM0KLpaeLNgM6XXIMsihQD0D8rX-"
subprocess.run(
    f"gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} "
    f"-O /kaggle/working/guarani_final/ --remaining-ok",
    shell=True
)
model_path = "/kaggle/working/guarani_final"
print(f"Model at: {model_path}")
print("Files:", os.listdir(model_path))

In [ ]:
ft_tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="spa_Latn")
ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path, torch_dtype=torch.float16
).to("cuda")
ft_model.eval()

# Load existing Spanish dev captions
with open(SPANISH_OUT) as f:
    spanish_captions = [l.strip() for l in f.readlines()]

hyps_ft = []
for cap in tqdm(spanish_captions, desc="Fine-tuned translation"):
    inputs = ft_tokenizer(
        cap, return_tensors="pt", truncation=True, max_length=128
    ).to("cuda")
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            forced_bos_token_id=ft_tokenizer.convert_tokens_to_ids("grn_Latn"),
            max_length=256,
            num_beams=4
        )
    hyps_ft.append(ft_tokenizer.decode(outputs[0], skip_special_tokens=True))

refs = df["target_caption"].tolist()[:len(hyps_ft)]
scores_ft = [CHRF(word_order=2).sentence_score(h, [r]).score
             for h, r in zip(hyps_ft, refs)]
media_ft = sum(scores_ft) / len(scores_ft)

print(f"Base NLLB:      {df['chrf_score'].mean():.2f}")
print(f"Fine-tuned:     {media_ft:.2f}")
print(f"Sheffield ref:  20.82")

# Verdict: base is slightly better on automatic metric.
# Fine-tuned was submitted based on expectation of better cultural coverage —
# confirmed by 3rd place in human evaluation.

## 7. Guaraní test set — Final submission

Runs Stage 1 (captioning) and Stage 2 (fine-tuned translation) on the test set.
Writes `guarani_submission.jsonl`.

In [ ]:
TEST_SPANISH_OUT   = f"{OUTPUT_DIR}/guarani_test_spanish_captions.txt"
TEST_TRANSLATED_OUT = f"{OUTPUT_DIR}/guarani_test_translated.txt"
TEST_JSONL          = "/kaggle/working/americasnlp2026/data/test/guarani/guarani.jsonl"
SUBMISSION_OUT      = f"{OUTPUT_DIR}/guarani_submission.jsonl"

test_df = pd.read_json(TEST_JSONL, lines=True)
test_df["filepath"] = test_df["filename"].apply(
    lambda x: f"/kaggle/working/americasnlp2026/data/test/guarani/{x}"
)
print(f"Test images: {len(test_df)}")

In [ ]:
# Stage 1: generate Spanish captions for test set
# (same logic as dev — resumes from existing progress)

# Reload VLM if needed
if "vlm" not in dir():
    processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
    vlm = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3-VL-8B-Instruct", torch_dtype=torch.float16, device_map="auto"
    )
    print(f"VLM loaded | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

if os.path.exists(TEST_SPANISH_OUT):
    with open(TEST_SPANISH_OUT) as f:
        generated_captions = [l.strip() for l in f.readlines()]
    print(f"Resuming from {len(generated_captions)}/{len(test_df)}")
else:
    generated_captions = []

for idx, row in tqdm(test_df.iloc[len(generated_captions):].iterrows(),
                     total=len(test_df)-len(generated_captions), desc="Captioning test"):
    try:
        img = PILImage.open(row["filepath"]).convert("RGB").resize((384, 384))
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": GUARANI_TEMPLATE_V2}
        ]}]
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = vlm.generate(
                **inputs, max_new_tokens=128,
                do_sample=False, repetition_penalty=1.3
            )
        response_text = processor.decode(
            outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
        )
    except Exception as e:
        print(f"Error on {row['id']}: {e}")
        response_text = "Imagen cultural guaraní."

    generated_captions.append(response_text)
    del inputs, outputs
    torch.cuda.empty_cache()

    with open(TEST_SPANISH_OUT, "w", encoding="utf-8") as f:
        for caption in generated_captions:
            f.write(caption.replace("\n", " ") + "\n")

print(f"Done: {len(generated_captions)} captions")

In [ ]:
# Stage 2: translate test set with fine-tuned model
del vlm, processor
torch.cuda.empty_cache(); gc.collect()

nllb_tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="spa_Latn")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path, torch_dtype=torch.float16
).to("cuda")

with open(TEST_SPANISH_OUT) as f:
    spanish_captions = [l.strip() for l in f.readlines()]

if os.path.exists(TEST_TRANSLATED_OUT):
    with open(TEST_TRANSLATED_OUT) as f:
        translated_captions = [l.strip() for l in f.readlines()]
    print(f"Resuming from {len(translated_captions)}/{len(spanish_captions)}")
else:
    translated_captions = []

for caption in tqdm(spanish_captions[len(translated_captions):], desc="Translating test"):
    inputs = nllb_tokenizer(
        caption, return_tensors="pt", truncation=True, max_length=512
    ).to("cuda")
    outputs = nllb_model.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("grn_Latn"),
        max_length=256
    )
    translated_captions.append(
        nllb_tokenizer.decode(outputs[0], skip_special_tokens=True)
    )
    with open(TEST_TRANSLATED_OUT, "w", encoding="utf-8") as f:
        for t in translated_captions:
            f.write(t + "\n")

print(f"Done: {len(translated_captions)} translations")

# Write submission JSONL
assert len(translated_captions) == len(test_df), (
    f"Mismatch: {len(translated_captions)} translations vs {len(test_df)} images"
)
with open(SUBMISSION_OUT, "w", encoding="utf-8") as f:
    for idx, row in test_df.iterrows():
        entry = row.to_dict()
        entry["predicted_caption"] = translated_captions[idx - test_df.index[0]]
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Submission written: {SUBMISSION_OUT} ({len(test_df)} entries)")

# Sample entry
with open(SUBMISSION_OUT) as f:
    print(json.dumps(json.loads(f.readline()), indent=2, ensure_ascii=False))

## 8. Maya, Wixárika, Nahuatl, Bribri — prompts and language codes

Cultural prompts for the remaining four languages.
Processed via `run_language()` in the next section.

In [ ]:
PROMPTS = {

    "maya": """Eres un sistema de subtitulado de imágenes para el pueblo Maya de México (Yucatán).
Genera un subtítulo en ESPAÑOL, conciso y culturalmente preciso (2-4 oraciones).

CONTEXTO CULTURAL:
- Henequén: fibra de agave, base de la economía yucateca histórica
- Huipil: vestido tradicional bordado con flores coloridas
- Milpa: sistema agrícola de maíz, frijol y calabaza
- Cenotes: pozos naturales sagrados para los mayas
- Chichen Itzá, Uxmal, Tulum: sitios arqueológicos mayas
- Hanal Pixán: día de muertos maya (noviembre)
- Jarana: danza tradicional yucateca
- Pib/Mucbipollo: tamal maya cocido en horno de tierra
- Sopa de Lima: sopa tradicional yucateca
- Cochinita Pibil: cerdo marinado en achiote cocido bajo tierra
- Xtabay: ser mitológico femenino maya
- Alux: duende maya guardián de los campos
- Chaac: dios maya de la lluvia
- Kukulkán: serpiente emplumada, deidad maya

DIRECTRICES:
1. Máximo 2-4 oraciones en español
2. Identifica el elemento cultural específico si lo reconoces
3. NO uses emojis ni caracteres especiales
4. Escribe SOLO en español""",

    "wixarika": """Eres un sistema de subtitulado de imágenes para el pueblo Wixárika (Huichol) de México.
Genera un subtítulo en ESPAÑOL, conciso y culturalmente preciso (2-4 oraciones).

CONTEXTO CULTURAL:
- Peyote (hikuri): cactus sagrado usado en ceremonias
- Wirikuta: sitio de peregrinación en San Luis Potosí
- Nierika: tabletas rituales con estambre en cera de abeja
- Cuadros de estambre: arte con visiones chamánicas, colores psicodélicos
- Arte con chaquira: cuentas sobre cera de abeja
- Ojo de Dios (tsikiri): símbolo de protección
- Mara'akame: chamán/cantador wixárika
- Kuchuri: morral bordado
- Rupurero: sombrero de palma decorado
- Kamirra/kutuni: camisa larga tradicional
- Juayame: faja ceremonial
- Tatewarí: Dios del Fuego/Sol
- Tatei Haramara: diosa del mar, sitio en San Blas Nayarit
- Ceremonias Mitote: rituales colectivos
- Nawá/tejuino: bebida ceremonial de maíz

DIRECTRICES:
1. Máximo 2-4 oraciones en español
2. Prefiere "Wixárika" sobre "Huichol"
3. NO uses emojis ni caracteres especiales
4. Escribe SOLO en español""",

    "nahuatl": """Eres un sistema de subtitulado de imágenes para el pueblo Nahua de México (Orizaba, Veracruz).
Genera un subtítulo en ESPAÑOL, conciso y culturalmente preciso (2-4 oraciones).

CONTEXTO CULTURAL:
- Mole, tamales, tlayudas: alimentos tradicionales nahuas
- Chinampas: jardines flotantes aztecas
- Teotihuacán, Tenochtitlán: ciudades prehispánicas
- Quetzalcóatl: serpiente emplumada, deidad mesoamericana
- Xochitl: flor sagrada en la cosmovisión nahua
- Día de Muertos: cempasúchil, altares, ofrendas
- Huipil: vestimenta tradicional femenina
- Copal: resina sagrada para rituales
- Temazcal: baño de vapor ceremonial
- Milpa: sistema agrícola tradicional
- Voladores de Papantla: ceremonia ritual nahua""",

    "bribri": """Eres un sistema de subtitulado de imágenes para el pueblo Bribri de Costa Rica.
Genera un subtítulo en ESPAÑOL, conciso y culturalmente preciso (2-4 oraciones).

CONTEXTO CULTURAL:
- Talamanca: territorio ancestral bribri (Cordillera de Talamanca)
- Cacao: planta sagrada para el pueblo bribri
- Sibö: Dios creador en la cosmovisión bribri
- Clanes matrilineales: estructura social bribri
- Sukia: chamán/sanador espiritual bribri
- Casa cónica circular (sulà): arquitectura tradicional bribri
- Chicha de maíz/pejibaye: bebida ceremonial
- Cestería: artesanía tradicional bribri
- Usure: ceremonia tradicional de muerte bribri
- Kéköldi: reserva indígena bribri""",
}

# NLLB target language codes
# NOTE: bzd_Latn and yua_Latn are missing from the base NLLB vocab (return id=3).
# Fine-tuned models learn them from training data.
LANG_CODES = {
    "guarani":  "grn_Latn",
    "maya":     "yua_Latn",
    "wixarika": "hch_Latn",
    "nahuatl":  "nah_Latn",
    "bribri":   "bzd_Latn",
}

# Paths to fine-tuned models (downloaded from Google Drive)
FINETUNED_MODEL_PATHS = {
    "guarani":  "/kaggle/working/guarani_final",
    "maya":     "/kaggle/working/maya_final",
    "wixarika": "/kaggle/working/wixarika_model/wixarika_final",
    "nahuatl":  "/kaggle/working/nahuatl_model/nahuatl_final",
    "bribri":   "/kaggle/working/bribri_model/bribri_final",
}

print("Prompts and language codes ready.")

## 9. `run_language()` — full pipeline for one language

Runs Stage 1 (VLM captioning) and Stage 2 (NMT translation) and writes the
submission JSONL. Resumes from existing progress if interrupted.

In [ ]:
import os, torch, json, gc, pandas as pd
from tqdm import tqdm
from PIL import Image as PILImage
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda"


def run_language(lang, prompt, tgt_lang_code):
    print(f"\n{'='*50}\nRUNNING: {lang.upper()}\n{'='*50}")

    TEST_JSONL     = f"/kaggle/working/americasnlp2026/data/test/{lang}/{lang}.jsonl"
    SPANISH_OUT    = f"/kaggle/working/{lang}_test_spanish.txt"
    TRANSLATED_OUT = f"/kaggle/working/{lang}_test_translated.txt"
    SUBMISSION_OUT = f"/kaggle/working/{lang}_submission.jsonl"

    test_df = pd.read_json(TEST_JSONL, lines=True)
    test_df["filepath"] = test_df["filename"].apply(
        lambda x: f"/kaggle/working/americasnlp2026/data/test/{lang}/{x}"
    )
    print(f"Loaded {len(test_df)} test images")

    # ── STAGE 1: VLM captioning ───────────────────────────────────────────────
    captions_complete = (
        os.path.exists(SPANISH_OUT)
        and len(open(SPANISH_OUT).readlines()) >= len(test_df)
    )

    if captions_complete:
        print("Stage 1: captions already complete, skipping")
        with open(SPANISH_OUT) as f:
            captions = [l.strip() for l in f.readlines()]
    else:
        captions = []
        if os.path.exists(SPANISH_OUT):
            with open(SPANISH_OUT) as f:
                captions = [l.strip() for l in f.readlines()]
        print(f"Stage 1: resuming from {len(captions)}/{len(test_df)}")

        processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
        vlm = AutoModelForImageTextToText.from_pretrained(
            "Qwen/Qwen3-VL-8B-Instruct",
            torch_dtype=torch.float16,
            device_map="auto",
        )

        with open(SPANISH_OUT, "a", encoding="utf-8") as out_f:
            for _, row in tqdm(
                test_df.iloc[len(captions):].iterrows(),
                total=len(test_df) - len(captions),
                desc=f"Captioning {lang}",
            ):
                try:
                    img = PILImage.open(row["filepath"]).convert("RGB").resize((384, 384))
                    messages = [{"role": "user", "content": [
                        {"type": "image", "image": img},
                        {"type": "text",  "text": prompt},
                    ]}]
                    inputs = processor.apply_chat_template(
                        messages, add_generation_prompt=True,
                        tokenize=True, return_dict=True, return_tensors="pt",
                    ).to(device)
                    with torch.no_grad():
                        outputs = vlm.generate(
                            **inputs,
                            do_sample=False,
                            max_new_tokens=128,
                            repetition_penalty=1.3,
                        )
                    caption = processor.decode(
                        outputs[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    ).strip()
                except Exception as e:
                    caption = ""
                    print(f"  Warning: {row['id']} — {e}")

                out_f.write(caption + "\n")
                captions.append(caption)
                del inputs, outputs
                torch.cuda.empty_cache()

        del vlm, processor
        torch.cuda.empty_cache(); gc.collect()
        print(f"Stage 1 complete: {len(captions)} captions")

    # ── STAGE 2: NMT translation ──────────────────────────────────────────────
    if (os.path.exists(TRANSLATED_OUT)
            and len(open(TRANSLATED_OUT).readlines()) >= len(test_df)):
        print("Stage 2: translations already complete, skipping")
        with open(TRANSLATED_OUT) as f:
            translations = [l.strip() for l in f.readlines()]
    else:
        print(f"Stage 2: translating {lang} → {tgt_lang_code}")

        model_path = FINETUNED_MODEL_PATHS.get(lang, "facebook/nllb-200-distilled-600M")
        nllb_tok = AutoTokenizer.from_pretrained(model_path, src_lang="spa_Latn")
        nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
            model_path, torch_dtype=torch.float16, low_cpu_mem_usage=True
        ).to("cuda")
        nllb_model.eval()

        bos_id = nllb_tok.convert_tokens_to_ids(tgt_lang_code)

        translations = []
        with open(TRANSLATED_OUT, "w", encoding="utf-8") as out_f:
            for caption in tqdm(captions, desc=f"Translating {lang}"):
                inputs = nllb_tok(
                    caption, return_tensors="pt",
                    truncation=True, max_length=64,
                ).to("cuda")
                with torch.no_grad():
                    output = nllb_model.generate(
                        **inputs,
                        forced_bos_token_id=bos_id,
                        max_length=256,
                        num_beams=4,
                        # Bribri: repetition penalty suppresses degenerate loops
                        # caused by small training data + morphological complexity
                        repetition_penalty=2.5 if lang == "bribri" else 1.0,
                        no_repeat_ngram_size=3 if lang == "bribri" else 0,
                    )
                t = nllb_tok.decode(output[0], skip_special_tokens=True)
                out_f.write(t + "\n")
                translations.append(t)
                del inputs, output

        del nllb_model, nllb_tok
        torch.cuda.empty_cache(); gc.collect()
        print(f"Stage 2 complete: {len(translations)} translations")

    # ── Write submission JSONL ────────────────────────────────────────────────
    with open(SUBMISSION_OUT, "w", encoding="utf-8") as out_f:
        for i, (_, row) in enumerate(test_df.iterrows()):
            entry = row.to_dict()
            entry["predicted_caption"] = translations[i] if i < len(translations) else ""
            out_f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Submission written: {SUBMISSION_OUT} ({len(test_df)} entries)")

## 10. Run Maya, Wixárika, Nahuatl, Bribri

In [ ]:
run_language("maya",     PROMPTS["maya"],     LANG_CODES["maya"])
run_language("wixarika", PROMPTS["wixarika"], LANG_CODES["wixarika"])
run_language("nahuatl",  PROMPTS["nahuatl"],  LANG_CODES["nahuatl"])
run_language("bribri",   PROMPTS["bribri"],   LANG_CODES["bribri"])

## 11. Dev set evaluation — all languages

Evaluates base NLLB and fine-tuned models on the dev set for
Wixárika, Nahuatl, Bribri, and Maya.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import CHRF
import pandas as pd, torch, os, gc

LANG_INFO = {
    "wixarika": {"spanish": f"{OUTPUT_DIR}/wixarika_dev_spanish.txt",  "tgt": "hch_Latn"},
    "nahuatl":  {"spanish": f"{OUTPUT_DIR}/nahuatl_dev_spanish.txt",   "tgt": "nah_Latn"},
    "bribri":   {"spanish": f"{OUTPUT_DIR}/bribri_dev_spanish.txt",    "tgt": "bzd_Latn"},
    "maya":     {"spanish": f"{OUTPUT_DIR}/maya_dev_spanish.txt",      "tgt": "yua_Latn"},
}

# Load base NLLB once for all languages
nllb_tok   = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
).to("cuda")

BASELINE_SCORES = {"wixarika": 2.48, "nahuatl": 3.29, "bribri": None, "maya": None}

for lang, info in LANG_INFO.items():
    out_path = f"{OUTPUT_DIR}/{lang}_dev_ours_translated.txt"
    if os.path.exists(out_path):
        with open(out_path) as f:
            done = f.readlines()
        print(f"{lang}: already {len(done)} lines, skipping")
        continue

    print(f"\nTranslating {lang} dev set...")
    if not os.path.exists(info["spanish"]):
        print(f"  Skipping — {info['spanish']} not found")
        continue

    with open(info["spanish"]) as f:
        spanish = [l.strip() for l in f.readlines()]

    dev_df = pd.read_json(
        f"{OUTPUT_DIR}/americasnlp2026/data/dev/{lang}/{lang}.jsonl",
        lines=True,
    )
    spanish = spanish[:len(dev_df)]

    translated = []
    for cap in tqdm(spanish, desc=lang):
        inputs = nllb_tok(cap, return_tensors="pt",
                          truncation=True, max_length=512).to("cuda")
        outputs = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tok.convert_tokens_to_ids(info["tgt"]),
            max_length=256,
        )
        translated.append(nllb_tok.decode(outputs[0], skip_special_tokens=True))
        del inputs, outputs

        with open(out_path, "w", encoding="utf-8") as f:
            for line in translated:
                f.write(line + "\n")

    refs   = dev_df["target_caption"].tolist()[:len(translated)]
    scores = [CHRF(word_order=2).sentence_score(h, [r]).score
              for h, r in zip(translated, refs)]
    mean   = sum(scores) / len(scores)
    base   = BASELINE_SCORES.get(lang)
    print(f"  {lang.upper()} base NLLB: {mean:.2f}"
          + (f" (baseline was {base:.2f})" if base else ""))

print("\nDev evaluation complete.")"

## 12. Verify all submission files

In [ ]:
import gc, torch

# Clear all models
for name in ["vlm", "processor", "nllb_model", "nllb_tok",
             "nllb_tokenizer", "ft_model", "ft_tokenizer", "model", "tokenizer"]:
    try:
        del globals()[name]
    except KeyError:
        pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")

print("\n--- Submission files ---")
for lang in ["guarani", "maya", "wixarika", "nahuatl", "bribri"]:
    path = f"{OUTPUT_DIR}/{lang}_submission.jsonl"
    if os.path.exists(path):
        n = sum(1 for _ in open(path))
        size = os.path.getsize(path) / 1024
        print(f"  OK  {lang}: {n} entries, {size:.0f} KB")
    else:
        print(f"  MISSING  {lang}")

## 13. Post-processing: loop detection and correction

Some languages (Wixárika, Guaraní) produced degenerate repetition loops in a subset of outputs. These cells detect and re-translate affected entries using more aggressive decoding parameters. The `is_genuine_loop()` function flags outputs where a single content token exceeds 40% of all tokens.

In [ ]:
import json
import torch
import os
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

def is_genuine_loop(text, loop_threshold=0.4):
    """Flag degenerate repetition: a single token > 40% of all tokens.
    Excludes common short function words to avoid false positives.
    """
    tokens = text.split()
    if len(tokens) < 10:
        return False
    function_words = {'in', 'yn', 'ha', 'pe', 'ko', 'the', 'a', 'ye',
                      'i', 'u', 'e', 'o', 'xu', 'tl', 'yt', 'my', 'ty', 'ra'}
    counts = Counter(tokens)
    top_token, top_count = counts.most_common(1)[0]
    if top_token.lower() in function_words:
        if len(counts) > 1:
            top_token, top_count = counts.most_common(2)[1]
        else:
            return top_count / len(tokens) > loop_threshold
    return (top_count / len(tokens)) > loop_threshold


def fix_loops(lang, submission_file, spanish_file, model_path, tgt_code, output_file):
    """Re-translate any entries where is_genuine_loop() returns True.
    Uses aggressive decoding (repetition_penalty=4.0, no_repeat_ngram_size=4).
    Falls back to sampling if beam search still produces a loop.
    """
    if not os.path.exists(submission_file):
        print(f"Submission not found: {submission_file}")
        return

    # Try alternate Spanish filename if primary not found
    if not os.path.exists(spanish_file):
        alt = spanish_file.replace(".txt", "_captions.txt")
        if os.path.exists(alt):
            spanish_file = alt
        else:
            print(f"Spanish captions not found: {spanish_file}")
            return

    with open(submission_file) as f:
        data = [json.loads(line) for line in f]
    with open(spanish_file, encoding="utf-8") as f:
        spanish_texts = [l.strip() for l in f]

    loop_indices = [
        i for i, entry in enumerate(data)
        if is_genuine_loop(entry.get("predicted_caption", ""))
    ]

    if not loop_indices:
        print(f"{lang}: no loops detected")
        return

    print(f"{lang}: {len(loop_indices)} loops detected — re-translating")

    tokenizer = AutoTokenizer.from_pretrained(
        "facebook/nllb-200-distilled-600M", src_lang="spa_Latn"
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path, torch_dtype=torch.float16, low_cpu_mem_usage=True
    ).to("cuda")
    model.eval()
    bos_id = tokenizer.convert_tokens_to_ids(tgt_code)

    beam_params = {
        "max_length": 80, "num_beams": 4,
        "repetition_penalty": 4.0, "no_repeat_ngram_size": 4,
        "length_penalty": 0.8, "early_stopping": True,
    }
    sample_params = {
        "max_length": 80, "do_sample": True,
        "temperature": 1.2, "top_p": 0.9,
        "repetition_penalty": 3.0, "no_repeat_ngram_size": 4,
    }

    for idx in tqdm(loop_indices, desc=f"Fixing {lang}"):
        if idx >= len(spanish_texts):
            continue
        inputs = tokenizer(
            spanish_texts[idx], return_tensors="pt",
            truncation=True, max_length=128
        ).to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=bos_id, **beam_params)
        new_text = tokenizer.decode(out[0], skip_special_tokens=True)

        # Fallback: sampling if beam search still loops
        if is_genuine_loop(new_text):
            with torch.no_grad():
                out = model.generate(**inputs, forced_bos_token_id=bos_id, **sample_params)
            new_text = tokenizer.decode(out[0], skip_special_tokens=True)

        data[idx]["predicted_caption"] = new_text
        torch.cuda.empty_cache()

    del model, tokenizer
    torch.cuda.empty_cache(); gc.collect()

    with open(output_file, "w", encoding="utf-8") as f:
        for entry in data:
            if "predicted_caption" not in entry and "target_caption" in entry:
                entry["predicted_caption"] = entry["target_caption"]
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    remaining = sum(1 for e in data if is_genuine_loop(e.get("predicted_caption", "")))
    print(f"{lang}: fixed. Residual loops: {remaining}. Saved: {output_file}")


# Fix Wixárika
fix_loops(
    lang            = "wixarika",
    submission_file = "/kaggle/working/wixarika_submission.jsonl",
    spanish_file    = "/kaggle/working/wixarika_test_spanish.txt",
    model_path      = "/kaggle/working/wixarika_final",
    tgt_code        = "hch_Latn",
    output_file     = "/kaggle/working/wixarika_FIXED.jsonl",
)

## 14. Guaraní: fix duplicate predictions

The Guaraní model occasionally produced the same output for multiple images (mode collapse). This cell re-translates duplicates using sampling with a unique seed per entry, ensuring all predictions are distinct.

In [ ]:
import json
import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

model_path = "/kaggle/working/guarani_final"
tokenizer  = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M", src_lang="spa_Latn"
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path, torch_dtype=torch.float16, low_cpu_mem_usage=True
).to("cuda")
model.eval()
bos_id = tokenizer.convert_tokens_to_ids("grn_Latn")

sub_file = "/kaggle/working/guarani_FINAL_submission.jsonl"
es_file  = "/kaggle/working/guarani_test_spanish_captions.txt"

with open(sub_file) as f:
    sub_data = [json.loads(line) for line in f]
with open(es_file) as f:
    spanish_texts = [l.strip() for l in f]

# Attach source caption to each entry
for i, entry in enumerate(sub_data):
    entry["source_caption"] = spanish_texts[i] if i < len(spanish_texts) else ""

# Find duplicated predictions — keep first, re-translate the rest
id_to_indices = defaultdict(list)
for i, entry in enumerate(sub_data):
    id_to_indices[entry.get("predicted_caption", "")].append(i)

to_retranslate = []
for pred, indices in id_to_indices.items():
    if len(indices) > 1:
        for rank, idx in enumerate(indices[1:], start=1):
            to_retranslate.append((idx, rank, sub_data[idx]))

print(f"Entries to re-translate: {len(to_retranslate)}")

# Re-translate with sampling + unique seed per entry
gen_params = {
    "max_length": 128, "do_sample": True,
    "temperature": 1.3, "top_p": 0.92,
    "repetition_penalty": 2.0, "no_repeat_ngram_size": 4,
}

for idx, rank, entry in tqdm(to_retranslate, desc="Re-translating duplicates"):
    seed = hash(entry["id"]) % 1000 + rank * 137
    torch.manual_seed(seed)
    spa_text = entry.get("source_caption", "")
    if not spa_text:
        continue
    inputs = tokenizer(
        [spa_text], return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, forced_bos_token_id=bos_id, **gen_params)
    sub_data[idx]["predicted_caption"] = tokenizer.decode(outputs[0], skip_special_tokens=True)

del model, tokenizer
torch.cuda.empty_cache(); gc.collect()

# Save clean file (remove helper source_caption field)
output_file = "/kaggle/working/guarani_FINAL_CLEAN.jsonl"
with open(output_file, "w") as f:
    for entry in sub_data:
        entry_clean = {k: v for k, v in entry.items() if k != "source_caption"}
        f.write(json.dumps(entry_clean, ensure_ascii=False) + "\n")

unique = len({e["predicted_caption"] for e in sub_data})
print(f"Total: {len(sub_data)}, Unique: {unique}")
print(f"Saved: {output_file}")

## 15. Assemble and verify final submission

Converts all language-specific JSONL files to the official submission format, verifies for loops and duplicates, then packages everything into a zip file.

In [ ]:
import json
import os
from collections import Counter

# Map each language to its final (post-processed) submission file
input_files = {
    "guarani": "/kaggle/working/guarani_FINAL_CLEAN.jsonl",
    "wixarika": "/kaggle/working/wixarika_FIXED.jsonl",
    "nahuatl":  "/kaggle/working/nahuatl_submission.jsonl",
    "bribri":   "/kaggle/working/bribri_submission.jsonl",
    "maya":     "/kaggle/working/maya_submission.jsonl",
}

output_dir = "/kaggle/working/final_submission"
os.makedirs(output_dir, exist_ok=True)

print("Converting to official submission format...")
print("="*60)

for lang, input_path in input_files.items():
    if not os.path.exists(input_path):
        print(f"MISSING  {lang}: {input_path}")
        continue

    with open(input_path) as f:
        entries = [json.loads(line) for line in f]

    # Convert to official format
    converted = []
    for entry in entries:
        new_entry = {
            "id":             entry.get("id", ""),
            "filename":       entry.get("filename", ""),
            "split":          entry.get("split", "test"),
            "culture":        entry.get("culture", lang),
            "language":       entry.get("language", ""),
            "iso_lang":       entry.get("iso_lang", ""),
            "target_caption": entry.get("predicted_caption",
                                        entry.get("target_caption", "")),
        }
        converted.append(new_entry)

    output_path = f"{output_dir}/{lang}.jsonl"
    with open(output_path, "w") as f:
        for entry in converted:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # Quick quality check
    preds   = [e["target_caption"] for e in converted]
    loops   = sum(1 for p in preds if is_genuine_loop(p))
    unique  = len(set(preds))
    empty   = sum(1 for p in preds if len(p.strip()) < 5)
    print(f"OK  {lang}: {len(converted)} entries | "
          f"unique={unique} | loops={loops} | empty={empty}")

print("="*60)

# Package into zip
import zipfile
zip_path = "/kaggle/working/USP_submission.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for lang in input_files:
        file_path = f"{output_dir}/{lang}.jsonl"
        if os.path.exists(file_path):
            z.write(file_path, f"{lang}.jsonl")
            print(f"Added: {lang}.jsonl")
        else:
            print(f"MISSING: {lang}.jsonl")

print(f"\nZip created: {zip_path}")
print(f"Size: {os.path.getsize(zip_path)/1024:.0f} KB")